# SimLab demo (slew, detumble, hold, eigenaxis, polhode)

Lightweight walkthrough for **Dynamics-calc**. It runs the SimLab CLI for `--scenario slew`, `--scenario detumble`, `--scenario hold` (identity hold under `EnvironmentalTorques`), `--scenario eigenaxis` (principal-axis slew, default LQR), and `--scenario polhode` (torque-free plant `sample_herpolhode` + energy–Casimir plots), then points at the committed recruiter figures (slew PNG/GIF from M1, Monte Carlo PNGs from PR #10, hold / eigenaxis pack, polhode / Casimir pair).

This notebook does **not** rewrite the plant, controllers, or estimators. Optional RKMK4 is a **library** switch on `step_rigid_body(..., method="rkmk4")` — it is **not** a SimLab CLI flag. Closed-loop runs stay on default RK4. MRP charts (`{stem}_mrp.png`) are **post-process** plots of \(\sigma\) reconstructed from logged \(q\). Polhode / Casimir charts call the existing plant helpers on logged \(\omega\).

**CWD:** repo root (`Dynamics-calc/`). Headless matplotlib is already `Agg` inside `attitude_sim.plots`.

```bash
python -m attitude_sim --list-scenarios
```

## Install (once)

```bash
python -m venv .venv
source .venv/bin/activate
pip install -e ".[dev]"
```

Jupyter is optional (`pip install jupyter` / VS Code / GitHub preview). The cells below only need the package CLI.

## Slew CLI

Recruiter-length run (writes `outputs/slew_summary.png` and `outputs/slew_attitude.gif`):

```bash
python -m attitude_sim --scenario slew
```

Regenerate the committed copies in `docs/figures/`:

```bash
python -m attitude_sim --scenario slew --out-dir docs/figures
```

The next cell is a **short** Agg PNG smoke (`--no-gif`, same policy as CI). It does not overwrite `docs/figures/`.

In [ ]:
import subprocess
import sys
from pathlib import Path

out = Path("outputs") / "demo"
out.mkdir(parents=True, exist_ok=True)
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "attitude_sim",
        "--scenario",
        "slew",
        "--t-final",
        "0.5",
        "--no-gif",
        "--out-dir",
        str(out),
    ]
)
print("wrote", out / "slew_summary.png")

**Committed rest-to-rest slew summary** (`docs/figures/slew_summary.png`). Default stack `pid` + `mekf`, 75°, 40 s.

![Rest-to-rest slew summary (pid + mekf)](../docs/figures/slew_summary.png)

**Committed attitude GIF** (`docs/figures/slew_attitude.gif`). CI does **not** smoke GIFs.

![Rest-to-rest slew attitude animation](../docs/figures/slew_attitude.gif)

## Detumble CLI

Named SimLab scenario (not part of the Monte Carlo harness):

```bash
python -m attitude_sim --scenario detumble
python -m attitude_sim --scenario detumble --out-dir outputs --no-gif
```

In [ ]:
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "attitude_sim",
        "--scenario",
        "detumble",
        "--t-final",
        "0.5",
        "--no-gif",
        "--out-dir",
        str(out),
    ]
)
print("wrote", out / "detumble_summary.png")

## Hold under EnvironmentalTorques

Named SimLab scenario. Default stack is **PID** + MEKF, identity command, demo-scale gravity-gradient + residual-dipole `EnvironmentalTorques` (plant input after the actuator; RK4 unchanged). The Monte Carlo harness can now sample `--scenario hold` (opt-in, keep N small).

Full-length run:

```bash
python -m attitude_sim --scenario hold
python -m attitude_sim --scenario hold --estimator truth --out-dir docs/figures --no-gif
```

`--no-env` disables the pack; `--env` enables it on any other scenario. The next cell is a short Agg PNG smoke (`--no-gif`). It writes `hold_summary.png`, `hold_mrp.png`, and `hold_env_torque.png`.

Committed copies (PID + truth, 30 s):

![Hold under EnvironmentalTorques](../docs/figures/hold_summary.png)

![Hold environmental torque](../docs/figures/hold_env_torque.png)

![Hold MRP attitude error](../docs/figures/hold_mrp.png)

In [ ]:
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "attitude_sim",
        "--scenario",
        "hold",
        "--t-final",
        "0.5",
        "--no-gif",
        "--out-dir",
        str(out),
    ]
)
print("wrote", out / "hold_summary.png")
print("wrote", out / "hold_mrp.png")
print("wrote", out / "hold_env_torque.png")

## Eigenaxis slew (LQR)

Rest-to-rest rotation about body \(z\) (a principal axis of the default \(J\)). The pack default controller is **LQR**; `--angle-deg` defaults to 30 (slew still defaults to 75). Override with `--controller pid` if you want the same ICs under PID.

```bash
python -m attitude_sim --scenario eigenaxis
python -m attitude_sim --scenario eigenaxis --estimator truth --out-dir docs/figures --no-gif
python -m attitude_sim --scenario eigenaxis --angle-deg 15 --estimator truth --no-gif
```

Short Agg PNG smoke (writes `eigenaxis_summary.png` and `eigenaxis_mrp.png`).

Committed copies (LQR + truth, 30°, 20 s):

![Eigenaxis LQR slew](../docs/figures/eigenaxis_summary.png)

![Eigenaxis MRP attitude error](../docs/figures/eigenaxis_mrp.png)

In [ ]:
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "attitude_sim",
        "--scenario",
        "eigenaxis",
        "--t-final",
        "0.5",
        "--no-gif",
        "--out-dir",
        str(out),
    ]
)
print("wrote", out / "eigenaxis_summary.png")
print("wrote", out / "eigenaxis_mrp.png")

## MRP post-process chart

`attitude_sim.plots.plot_mrp_error` reconstructs the shadow-switched attitude-error MRP \(\sigma(q_{\mathrm{des}}^{\ast}\otimes q)\) from the quaternion log. The plant integrator is unchanged (`x=[q,ω]`). `--mrp-plot` is on by default for `hold` / `eigenaxis`; pass it explicitly to add `{stem}_mrp.png` to slew or detumble.

Optional programmatic helper (same Agg backend):

```python
from pathlib import Path
from attitude_sim.plots import plot_mrp_error
from attitude_sim.sim import make_scenario_config, run_slew

log = run_slew(make_scenario_config("eigenaxis", t_final=0.5, plot=False, gif=False, estimator="truth"))
plot_mrp_error(log, Path("outputs/demo/eigenaxis_mrp_from_log.png"))
```

Regenerate committed hold / eigenaxis copies (overwrites `docs/figures/`):

```bash
python -m attitude_sim --scenario hold --estimator truth --out-dir docs/figures --no-gif
python -m attitude_sim --scenario eigenaxis --out-dir docs/figures --no-gif
```

## Torque-free polhode / energy–Casimir

Open-loop SimLab scenario. Calls the plant helper `sample_herpolhode` (no controller, no estimator, env forced off) and writes `{stem}_polhode.png` (body \(\omega(t)\) + algebraic intersection + herpolhode) and `{stem}_casimir.png` (\(T\), \(|h|^2\), residuals). `--polhode-plot` applies the same post-process helpers to any logged \(\omega\).

```bash
python -m attitude_sim --scenario polhode
python -m attitude_sim --scenario polhode --out-dir docs/figures --no-gif
python -m attitude_sim --scenario slew --polhode-plot --t-final 0.5 --no-gif
```

Short Agg PNG smoke (writes `polhode_summary.png`, `polhode_polhode.png`, `polhode_casimir.png`).

Committed copies (15 s, stock smallsat \(J\), around-min \(\omega_0\)):

![Torque-free polhode / herpolhode](../docs/figures/polhode_polhode.png)

![Energy–Casimir first integrals](../docs/figures/polhode_casimir.png)

![Polhode summary](../docs/figures/polhode_summary.png)


In [ ]:
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "attitude_sim",
        "--scenario",
        "polhode",
        "--t-final",
        "0.5",
        "--no-gif",
        "--out-dir",
        str(out),
    ]
)
print("wrote", out / "polhode_summary.png")
print("wrote", out / "polhode_polhode.png")
print("wrote", out / "polhode_casimir.png")


## Monte Carlo figures (PR #10)

`python -m attitude_sim.monte_carlo` **defaults to slew**. Opt-in `--scenario hold|eigenaxis` and `--env` reuse the same `run_slew` path (CI keeps N tiny). Committed summary PNGs remain the N=40 slew sweep:

- `docs/figures/mc_final_att_error_hist.png` — final geodesic attitude-error histogram
- `docs/figures/mc_settle_vs_noise.png` — settle-time proxy vs log-uniform sensor-noise scale

![Monte Carlo final attitude-error histogram](../docs/figures/mc_final_att_error_hist.png)

![Monte Carlo settle time vs sensor-noise scale](../docs/figures/mc_settle_vs_noise.png)

Named-scenario / env-on expansions (keep N small):

```bash
python -m attitude_sim.monte_carlo --n 20 --scenario eigenaxis --estimator truth
python -m attitude_sim.monte_carlo --n 20 --scenario hold --estimator truth
python -m attitude_sim.monte_carlo --n 20 --scenario slew --env --estimator truth
```

Regenerate the committed copies (overwrites `docs/figures/mc_*.png`):

```bash
python -m attitude_sim.monte_carlo --n 40 --seed 0 --estimator mekf \
    --t-final 40 --inertia-frac 0.05 \
    --plot --out-dir docs/figures \
    --json outputs/mc_slew.json --csv outputs/mc_slew.csv
```

Re-plot from a previous JSON (no new trials):

```bash
python -m attitude_sim.monte_carlo --from-json outputs/mc_slew.json \
    --plot --out-dir docs/figures
```

CI covers the Agg PNG helpers (`tests/test_monte_carlo.py`) plus a check that these files exist (`tests/test_docs_figures.py`). GIF smoke is skipped.